# 05 — Feature Engineering Candidates

**Amaç:** Ratio, missing, outlier, interaction ve rule-based feature adaylarını target ile değerlendirmek.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## Aday feature setini üret

In [ ]:
from kaggle_s6_e7.features import (add_ratio_features,add_missing_features,add_categorical_interactions,
    add_rule_features,fit_outlier_bounds,add_outlier_flags)
feature_cols=[c for c in train if c not in [ID_COL,TARGET_COL]]
train_fe=add_ratio_features(train)
train_fe=add_missing_features(train_fe,feature_cols)
train_fe=add_categorical_interactions(train_fe)
train_fe=add_rule_features(train_fe)
ratio_cols=["calorie_per_step","calorie_per_exercise_min","step_per_exercise_min",
            "water_per_bmi","exercise_per_bmi","steps_per_sleep_hour"]
bounds=fit_outlier_bounds(train_fe,num_cols+ratio_cols)
train_fe=add_outlier_flags(train_fe,bounds)
print("Yeni kolon sayısı:",len(train_fe.columns)-len(train.columns))

## Ratio dağılımları ve target ilişkisi

In [ ]:
ratio_summary=train_fe.groupby(TARGET_COL)[ratio_cols].describe(percentiles=[.01,.05,.5,.95,.99]).T
display(ratio_summary); ratio_summary.to_csv(TABLES_DIR / "05_ratio_target_summary.csv")
plot_fe=train_fe.sample(min(PLOT_SAMPLE_SIZE,len(train_fe)),random_state=RANDOM_STATE)
fig,axes=plt.subplots(3,2,figsize=(12,12))
for col,ax in zip(ratio_cols,axes.flat):
    sns.boxplot(data=plot_fe,x=TARGET_COL,y=col,showfliers=False,ax=ax)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "05_ratio_by_target.png",dpi=150); plt.show()

## Interaction target tabloları

In [ ]:
interaction_cols=["stress_sleep_quality","activity_diet","smoking_activity","gender_activity"]
interaction_tables=[]
for col in interaction_cols:
    table=pd.crosstab(train_fe[col],train_fe[TARGET_COL],normalize="index")
    table.insert(0,"feature",col); interaction_tables.append(table.reset_index())
    print(f"\n{col}"); display(table)
pd.concat(interaction_tables,ignore_index=True).to_csv(TABLES_DIR / "05_interaction_target.csv",index=False)

## Rule-based flag target tabloları

In [ ]:
rule_cols=["low_sleep_flag","high_sleep_flag","high_bmi_flag","low_bmi_flag",
           "high_heart_rate_flag","low_heart_rate_flag","low_steps_flag","high_steps_flag"]
rule_tables=[]
for col in rule_cols:
    table=pd.crosstab(train_fe[col],train_fe[TARGET_COL],normalize="index")
    table.insert(0,"feature",col); rule_tables.append(table.reset_index())
rule_target=pd.concat(rule_tables,ignore_index=True)
display(rule_target); rule_target.to_csv(TABLES_DIR / "05_rule_target.csv",index=False)

## Preprocessing karar matrisi

| Paket | İçerik | Karar amacı |
|---|---|---|
| V1 | Median, `missing`, missing flag | Güvenli baseline |
| V2 | V1 + ratio, count, interaction, outlier flag | İlk favori |
| V3 | V2 + train %0.1–%99.9 clipping | Ayrı deney |

Her adayı yalnız target ayrışması, train/test stabilitesi ve yorumlanabilirlik birlikte destekliyorsa `eda_findings.md` içinde kabul et.